
Input: `outputs/step3_models.joblib`.
Output: `outputs/rf_importance.csv`, figure `slide_figures/perm.png`.

In [ ]:
# Working folder: holds HABs_ScrippsPier_v3.xlsx and the outputs/ folder shared by all steps.
# In Colab the folder is on Google Drive, so every step notebook can read what the previous step saved.
import os
WORK_FOLDER = "/content/drive/MyDrive/BIF633_Progress2"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(WORK_FOLDER, exist_ok=True)
    os.chdir(WORK_FOLDER)
except ImportError:
    pass  # not running in Colab: use the current folder
os.makedirs("outputs", exist_ok=True)
print("Working folder:", os.getcwd())

In [ ]:
# Load everything saved by the previous step
import joblib
globals().update(joblib.load("outputs/step3_models.joblib"))
print("Loaded outputs/step3_models.joblib")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

# The models predict log(1 + Chl-a). Convert back to mg/m3 before scoring.
def to_mg(log_values):
    return np.clip(np.expm1(np.asarray(log_values)), 0, None)  # same as the team's convert_log_prediction_to_original

def rmse(observed, predicted):
    return np.sqrt(mean_squared_error(observed, predicted))

def mae(observed, predicted):
    return mean_absolute_error(observed, predicted)

# Observed Chl-a (mg/m3) for the training and validation weeks
observed_train = to_mg(y_train)
observed_validation = to_mg(y_validation)

# The three core models and the feature set used for "set C" by each model
core_models = ["Linear Regression", "Random Forest", "SVR"]
set_C_name = {
    "Linear Regression": "C_Linear_Selected",   # 13 features kept after the VIF check
    "Random Forest": "C_Full_Temporal",
    "SVR": "C_Full_Temporal",
}
short_name = {
    "A_Environmental": "A",
    "B_Environmental_Seasonal": "B",
    "C_Full_Temporal": "C",
    "C_Linear_Selected": "C",
}
print("Ready")

In [ ]:
rf = trained_models["Random Forest__C_Full_Temporal"]["best_estimator"]
features = processed_feature_sets["C_Full_Temporal"]["features"]
X_validation_C = processed_feature_sets["C_Full_Temporal"]["X_validation"]

permutation = permutation_importance(rf, X_validation_C, y_validation,
                                     n_repeats=30, random_state=42,
                                     scoring="neg_root_mean_squared_error")

importance = pd.DataFrame({
    "Feature": features,
    "Impurity (training)": rf.feature_importances_,
    "Permutation (validation)": permutation.importances_mean,
    "Permutation SD": permutation.importances_std,
}).sort_values("Permutation (validation)", ascending=False)
display(importance.round(4))

importance_sorted = importance.sort_values("Permutation (validation)")
plt.figure(figsize=(7, 6))
plt.barh(importance_sorted["Feature"], importance_sorted["Permutation (validation)"],
         xerr=importance_sorted["Permutation SD"], color="teal")
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("Rise in validation error when shuffled")
plt.tight_layout()
plt.show()

In [ ]:
importance.to_csv("outputs/rf_importance.csv", index=False)
print("Saved outputs/rf_importance.csv")

In [ ]:
import os
import matplotlib.dates as mdates
os.makedirs("slide_figures", exist_ok=True)  # figures go here
plt.rcParams.update({"font.size": 18, "axes.spines.top": False, "axes.spines.right": False})
PLUM, TEAL, OLIVE, GREY = "#8A3560", "#00958F", "#A4983A", "#B3B0AA"


# Slide 7 (right): permutation importance of the Random Forest
colour = [PLUM if f.startswith("Chloro") else (OLIVE if f.startswith("month") else TEAL)
          for f in importance_sorted["Feature"]]
fig, ax = plt.subplots(figsize=(8.1, 6.7))
ax.barh(importance_sorted["Feature"], importance_sorted["Permutation (validation)"],
        xerr=importance_sorted["Permutation SD"], color=colour, height=0.66,
        error_kw={"ecolor": "#555555", "elinewidth": 1})
ax.axvline(0, color="black", linewidth=1)
ax.tick_params(axis="y", length=0, labelsize=16)
ax.tick_params(axis="x", labelsize=15)
ax.xaxis.set_major_locator(plt.MaxNLocator(5))
ax.spines["left"].set_visible(False)
ax.xaxis.grid(True, color="#E4E1DC")
ax.set_axisbelow(True)
ax.set_xlabel("Rise in validation error when shuffled")
legend_boxes = [plt.Rectangle((0, 0), 1, 1, color=c) for c in (TEAL, OLIVE, PLUM)]
ax.legend(legend_boxes, ["Environment", "Season", "Chl-a history"], frameon=False,
          loc="lower left", bbox_to_anchor=(-0.02, 1.0), ncol=3, fontsize=15,
          handlelength=1.2, columnspacing=1.0)
plt.tight_layout()
plt.savefig("slide_figures/perm.png", dpi=200)
plt.show()